<a href="https://colab.research.google.com/github/ZahraRasooli-200/MATLAB-practice/blob/main/cat_dog_ResNET34.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import os
import tarfile
import urllib.request
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
import matplotlib.pyplot as plt

In [ ]:
https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz

In [9]:
#function to download and extract dataset
def download_and_extract(url, dest):
    if not os.path.exists(dest):
        os.makedirs(dest)
    file_name = url.split('/')[-1]
    file_path = os.path.join(dest, file_name)
    if not os.path.exists(file_path):
        print(f"Downloading {url}...")
        urllib.request.urlretrieve(url, file_path)
    print(f"Extracting {file_path}...")
    with tarfile.open(file_path) as tar:
        tar.extractall(dest)

In [10]:
root = './data'
images_url = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz'
annotations_url = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz'
download_and_extract(images_url, root)
download_and_extract(annotations_url, root)

Extracting ./data/images.tar.gz...


/tmp/ipykernel_579/1755157291.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest)


Extracting ./data/annotations.tar.gz...


In [14]:
def get_images_labels(images_path, annotations_path):
    images = []
    labels = []
    class_map = {
        'Persian': 0,
        'shiba_inu': 1,
        'saint_bernard': 2,
    }
    with open(annotations_path) as f:
        lines = f.readlines()
    for line in lines:
        img_name = line.strip().split()[0]
        breed_name = img_name.rsplit("_",1)[0]
        if breed_name in class_map:
            img_path = os.path.join(images_path, f'{img_name}.jpg')
            if os.path.exists(img_path):
                images.append(Image.open(img_path).convert('RGB'))
                labels.append(class_map[breed_name])
    return images, labels

In [15]:
#split using Pytorch random_split on indices
images, labels = get_images_labels(images_path='./data/images', annotations_path='./data/annotations/trainval.txt')
indices = list(range(len(images)))
train_indices, val_indices = random_split(indices,[0.8,0.2],generator=torch.Generator().manual_seed(42))

train_paths = [images[i] for i in train_indices]
train_labels = [labels[i] for i in train_indices]
val_path = [images[i] for i in val_indices]
val_labels = [labels[i] for i in val_indices]

In [16]:
class CustomPetDataset(Dataset):
  def __init__(self, images, labels, transform):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image = self.images[idx]
    #image = Image.open(img_path).convert('RGB')
    label = self.labels[idx]
    image = self.transform(image)
    return image, label

In [17]:
#Data transformation (using RGB,larger size,and standard augmentations)
train_transform = transforms.Compose([
    transforms.Resize((256)), #128x256x3 --> 256x512x3
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), #3x224x224\
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [18]:
#Create separate datasets with split lists and transforms
train_dataset = CustomPetDataset(train_paths, train_labels, transform=train_transform)
val_dataset = CustomPetDataset(val_path, val_labels, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

In [24]:
#Device configuration
Device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [20]:
model = models.resnet34(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 123MB/s]


In [21]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [25]:
for param in model.parameters():
  param.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 3)
model = model.to(Device)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001, weight_decay=0.0001)
criterion = nn.CrossEntropyLoss()

In [35]:
model = models.resnet34(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [36]:
for param in model.fc.parameters():
  param.requires_grad = False

for param in model.layer4.parameters():
  param.requires_grad = True

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 3)
model = model.to(Device)

params_to_update = [
    {"params": model.fc.parameters(), "lr": 0.001},
    {"params": model.layer4.parameters(), "lr": 0.00001}
]

optimizer = torch.optim.Adam(params_to_update, weight_decay=0.0001)
criterion = nn.CrossEntropyLoss()

In [29]:
def train_epoch(model, loader, criterion, optimizer, device):
  model.train()
  train_loss = 0.0
  corrects = 0
  total = 0
  for images, targets in loader:
    images = images.to(device)
    targets = targets.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
    predictions = outputs.argmax(dim=1)
    corrects += (predictions == targets).sum().item()
    total += targets.size(0)
  train_loss /= len(loader)
  train_acc = corrects / total

  return train_loss, train_acc


In [33]:
def val_epoch(model, loader, criterion, device):
  model.eval()
  val_loss = 0.0
  corrects = 0
  total = 0
  with torch.no_grad():
    for images, targets in loader:
      images = images.to(device)
      targets = targets.to(device)
      outputs = model(images)
      loss = criterion(outputs, targets)
      val_loss += loss.item()
      predictions = outputs.argmax(dim=1)
      corrects += (predictions == targets).sum().item()
      total += targets.size(0)
    val_loss /= len(loader)
    train_acc = corrects / total

    return train_loss, train_acc


In [34]:
#BUILD MODEL

epochs = 100
best_acc = 0.0
for epoch in range(epochs):
  train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, Device)
  val_loss, val_acc = val_epoch(model, val_loader, criterion, Device)
  if val_acc > best_acc:
    best_acc = val_acc
    torch.save(model.state_dict(), 'best_model.pth')
  print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

Epoch 1/100, Train Loss: 0.3490, Train Acc: 0.9375, Val Loss: 0.3490, Val Acc: 1.0000
Epoch 2/100, Train Loss: 0.2256, Train Acc: 0.9458, Val Loss: 0.2256, Val Acc: 1.0000
Epoch 3/100, Train Loss: 0.1624, Train Acc: 0.9667, Val Loss: 0.1624, Val Acc: 1.0000
Epoch 4/100, Train Loss: 0.1172, Train Acc: 0.9792, Val Loss: 0.1172, Val Acc: 1.0000
Epoch 5/100, Train Loss: 0.1443, Train Acc: 0.9708, Val Loss: 0.1443, Val Acc: 1.0000
Epoch 6/100, Train Loss: 0.0887, Train Acc: 0.9875, Val Loss: 0.0887, Val Acc: 1.0000
Epoch 7/100, Train Loss: 0.1172, Train Acc: 0.9750, Val Loss: 0.1172, Val Acc: 1.0000
Epoch 8/100, Train Loss: 0.0737, Train Acc: 0.9958, Val Loss: 0.0737, Val Acc: 1.0000
Epoch 9/100, Train Loss: 0.0605, Train Acc: 0.9917, Val Loss: 0.0605, Val Acc: 1.0000
Epoch 10/100, Train Loss: 0.0665, Train Acc: 0.9708, Val Loss: 0.0665, Val Acc: 1.0000
Epoch 11/100, Train Loss: 0.0565, Train Acc: 0.9833, Val Loss: 0.0565, Val Acc: 1.0000
Epoch 12/100, Train Loss: 0.0531, Train Acc: 0.9917,

In [39]:
#BUILD MODEL

epochs = 100
best_acc = 0.0
for epoch in range(epochs):
  train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, Device)
  val_loss, val_acc = val_epoch(model, val_loader, criterion, Device)
  if val_acc > best_acc:
    best_acc = val_acc
    torch.save(model.state_dict(), 'best_model.pth')
  print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

Epoch 1/100, Train Loss: 0.0014, Train Acc: 1.0000, Val Loss: 0.0014, Val Acc: 1.0000
Epoch 2/100, Train Loss: 0.0011, Train Acc: 1.0000, Val Loss: 0.0011, Val Acc: 1.0000
Epoch 3/100, Train Loss: 0.0018, Train Acc: 1.0000, Val Loss: 0.0018, Val Acc: 1.0000
Epoch 4/100, Train Loss: 0.0007, Train Acc: 1.0000, Val Loss: 0.0007, Val Acc: 1.0000
Epoch 5/100, Train Loss: 0.0007, Train Acc: 1.0000, Val Loss: 0.0007, Val Acc: 1.0000
Epoch 6/100, Train Loss: 0.0024, Train Acc: 1.0000, Val Loss: 0.0024, Val Acc: 1.0000
Epoch 7/100, Train Loss: 0.0015, Train Acc: 1.0000, Val Loss: 0.0015, Val Acc: 1.0000
Epoch 8/100, Train Loss: 0.0014, Train Acc: 1.0000, Val Loss: 0.0014, Val Acc: 1.0000
Epoch 9/100, Train Loss: 0.0055, Train Acc: 0.9958, Val Loss: 0.0055, Val Acc: 1.0000
Epoch 10/100, Train Loss: 0.0007, Train Acc: 1.0000, Val Loss: 0.0007, Val Acc: 1.0000
Epoch 11/100, Train Loss: 0.0007, Train Acc: 1.0000, Val Loss: 0.0007, Val Acc: 1.0000
Epoch 12/100, Train Loss: 0.0009, Train Acc: 1.0000,

In [40]:
#LOAD BEST MODEL
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  